# 🧹 Limpieza y Transformación
## Foco: Efectos Secundarios por Componente

**Objetivo:** Convertir el dataset de nivel-medicamento a nivel-componente × efecto.  
Este es el paso clave que habilita todo el análisis posterior.

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../src/enfoque3')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from src.load_data import load_medicine_data
from cleaning import clean_data, split_side_effects, split_components
from transform import explode_effects, explode_all, build_effects_matrix, build_component_effect_crosstab
from validation import full_quality_report

Path("../outputs/figures").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

df_raw = load_medicine_data(download_if_missing=True)
print(f"Datos cargados: {df_raw.shape}")
df_raw.head(3)

## 1. Decisión técnica: ¿Cómo separamos los efectos secundarios?

La columna `Side_effects` **no usa comas**. Los efectos están concatenados usando **mayúscula inicial** como delimitador implícito.

**Solución:** `re.findall(r'[A-Z][^A-Z]*', texto)`  
Captura cada secuencia que comienza con mayúscula, preservando efectos multi-palabra como "Alta presión sanguínea".

In [ ]:
# Demostración de la decisión de limpieza
ejemplo = "Vomiting Nausea High blood pressure Dry skin"
resultado = split_side_effects(ejemplo)
print(f"Input : '{ejemplo}'")
print(f"Output: {resultado}")

print()

ejemplo_real = df_raw["Side_effects"].iloc[0]
print(f"Ejemplo real del dataset:")
print(f"Input : '{ejemplo_real}'")
print(f"Output: {split_side_effects(ejemplo_real)}")

## 2. Aplicar pipeline de limpieza

In [ ]:
# Aplicar limpieza completa
df_clean = clean_data(df_raw)

print(f"Shape original : {df_raw.shape}")
print(f"Shape limpio   : {df_clean.shape}")
print(f"\nColumnas nuevas generadas:")
df_clean[["Medicine Name", "componentes", "efectos_secundarios", "n_efectos", "n_componentes"]].head(3)

In [ ]:
print("=== EFECTOS SECUNDARIOS ===")
print(f"Media por medicamento  : {df_clean['n_efectos'].mean():.2f}")
print(f"Mediana                : {df_clean['n_efectos'].median():.2f}")
print(f"Máximo                 : {df_clean['n_efectos'].max()}")
print(f"Medicamentos sin efecto: {(df_clean['n_efectos'] == 0).sum()}")

print("\n=== COMPONENTES ===")
print(f"Media por medicamento  : {df_clean['n_componentes'].mean():.2f}")
print(f"Mediana                : {df_clean['n_componentes'].median():.2f}")
print(f"Máximo                 : {df_clean['n_componentes'].max()}")
print(f"Medicamentos sin comp. : {(df_clean['n_componentes'] == 0).sum()}")

### Interpretación
- En promedio cada medicamento tiene **6.92 efectos secundarios**, con un máximo de 36.
- La mediana es 6, lo que indica que la distribución es levemente asimétrica hacia valores altos.
- Ningún medicamento quedó sin efectos secundarios tras la limpieza 
- En promedio cada medicamento tiene **1.53 componentes activos**, siendo 1 el caso más común.
- El máximo de 9 componentes indica medicamentos de composición compleja.
- Ningún medicamento quedó sin componentes tras la limpieza 

## 3. Transformación clave: explode()

El dataset está a nivel de **medicamento**. Para analizar la relación componente → efecto, necesitamos una fila por cada par (componente, efecto).

**`explode_all()`** convierte esto:

| Medicamento | Componentes | Efectos |
|-------------|-------------|---------|
| Med1        | [A, B]      | [náusea, dolor] |

En esto:

| Medicamento | Componente | Efecto |
|-------------|------------|--------|
| Med1        | A          | náusea |
| Med1        | A          | dolor  |
| Med1        | B          | náusea |
| Med1        | B          | dolor  |

Sin esta transformación **no es posible** analizar qué componente específico está asociado a qué efecto.

In [28]:
# Transformación explode
df_exploded = explode_all(df_clean)

print(f"Shape antes del explode : {df_clean.shape}")
print(f"Shape después del explode: {df_exploded.shape}")
print(f"\nEjemplo — Augmentin 625 Duo Tablet expandido:")
df_exploded[df_exploded["Medicine Name"] == "Augmentin 625 Duo Tablet"][
    ["Medicine Name", "componentes", "efectos_secundarios"]
]

Shape antes del explode : (11825, 14)
Shape después del explode: (124349, 14)

Ejemplo — Augmentin 625 Duo Tablet expandido:


,Medicine Name,componentes,efectos_secundarios
9,Augmentin 625 Duo Tablet,amoxycillin,vomiting
10,Augmentin 625 Duo Tablet,amoxycillin,nausea
11,Augmentin 625 Duo Tablet,amoxycillin,diarrhea
12,Augmentin 625 Duo Tablet,amoxycillin,mucocutaneous candidiasis
13,Augmentin 625 Duo Tablet,clavulanic acid,vomiting
14,Augmentin 625 Duo Tablet,clavulanic acid,nausea
15,Augmentin 625 Duo Tablet,clavulanic acid,diarrhea
16,Augmentin 625 Duo Tablet,clavulanic acid,mucocutaneous candidiasis


### Interpretación
- El dataset pasó de **11.825 filas** a **124.349 filas** tras el explode.
- Cada fila ahora representa un par único **(componente, efecto)** por medicamento.
- El ejemplo de Augmentin lo ilustra perfectamente: tenía 2 componentes y 4 efectos → generó 8 filas (producto cartesiano).
- Esta estructura es la que permite responder: **¿qué componente está asociado a qué efecto?**